# Паттерн 2: Hierarchical Supervisor — дерево управления

Когда один супервайзер управляет семью и более агентами, он начинает ошибаться в маршрутизации — слишком много вариантов для одного LLM-вызова. Решение — иерархия супервайзеров: верхний координатор делегирует задачи командам, а внутри каждой команды свой локальный супервайзер (или конвейер) управляет специалистами. Каждая команда — скомпилированный подграф, который родительский граф видит как чёрный ящик.

```mermaid
graph LR
    START((START)) --> top_supervisor{{"🎯 Top Supervisor<br/><i>распределяет задачи</i>"}}
    top_supervisor -->|research_team| RT
    top_supervisor -->|writing_team| WT
    top_supervisor -->|FINISH| END((END))
    RT --> top_supervisor
    WT --> top_supervisor

    subgraph Research Team
        RT{{"🎯 RT Supervisor<br/><i>координирует группу</i>"}} --> web(["🔹 web_search<br/><i>выполняет работу</i>"])
        RT --> doc(["🔹 doc_search<br/><i>выполняет работу</i>"])
        web --> RT
        doc --> RT
        RT -->|FINISH| RT_F(["🔹 finalizer<br/><i>выполняет работу</i>"])
    end

    subgraph Writing Team
        WT{{"🎯 WT Supervisor<br/><i>координирует группу</i>"}} --> drafter(["🔹 drafter<br/><i>выполняет работу</i>"])
        drafter --> editor(["🔹 editor<br/><i>выполняет работу</i>"])
    end

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef subgraphStyle fill:#FDEBD0,stroke:#E67E22,stroke-width:2px,color:#333

    class top_supervisor,RT,WT coordinator
    class web,doc,RT_F,drafter,editor worker
    class START,END terminal
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

llm = get_llm()

API key is set


## Подграф 1: Исследовательская команда

Каждая команда — это полноценный скомпилированный граф со своим состоянием и внутренней логикой. Родительский граф ничего не знает о том, что происходит внутри: он передаёт задачу и получает результат. `ResearchState` содержит поле `task` (что исследовать) и `research` (накопленные факты). Два агента работают последовательно: веб-исследователь находит факты, документалист дополняет их техническими деталями.

In [3]:
class ResearchState(TypedDict):
    task: str
    research: str


def web_researcher(state: ResearchState) -> dict:
    """Агент: поиск информации в вебе."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — веб-исследователь. Найди 3-4 ключевых факта по теме. "
            "Кратко, по пунктам. Опирайся на общедоступную информацию."
        )),
        HumanMessage(content=state["task"])
    ])
    print(f"  [Веб-исследователь] Факты найдены ({len(response.content)} символов)")
    return {"research": response.content}


def doc_researcher(state: ResearchState) -> dict:
    """Агент: поиск в документации / внутренних источниках."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — документалист. Дополни существующее исследование "
            "техническими деталями: определения, стандарты, спецификации. "
            "Кратко: 2-3 пункта, не повторяй уже найденное."
        )),
        HumanMessage(content=f"Тема: {state['task']}\n\nУже найдено:\n{state['research']}")
    ])
    combined = f"{state['research']}\n\n[Документалист]:\n{response.content}"
    print(f"  [Документалист] Дополнено ({len(response.content)} символов)")
    return {"research": combined}


research_graph = StateGraph(ResearchState)
research_graph.add_node("web", web_researcher)
research_graph.add_node("doc", doc_researcher)
research_graph.add_edge(START, "web")
research_graph.add_edge("web", "doc")
research_graph.add_edge("doc", END)
research_compiled = research_graph.compile()

## Подграф 2: Писательская команда

Писательская команда устроена проще — это линейный конвейер из двух узлов: автор (`drafter`) пишет черновик на основе собранных фактов, затем редактор (`editor`) улучшает стиль и читаемость. Принцип тот же: собственное состояние `WritingState`, компиляция в отдельный граф.

In [4]:
class WritingState(TypedDict):
    task: str
    research: str
    draft: str
    final_text: str


def drafter(state: WritingState) -> dict:
    """Агент: пишет черновик статьи."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — автор. Напиши связный текст (5-6 предложений) "
            "на основе предоставленных фактов. Не выдумывай."
        )),
        HumanMessage(content=f"Тема: {state['task']}\n\nФакты:\n{state['research']}")
    ])
    print(f"  [Автор] Черновик готов ({len(response.content)} символов)")
    return {"draft": response.content}


def editor(state: WritingState) -> dict:
    """Агент: редактирует и улучшает черновик."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — редактор. Улучши текст: стиль, логика, читаемость. "
            "Верни финальную версию. Не добавляй новых фактов."
        )),
        HumanMessage(content=state["draft"])
    ])
    print(f"  [Редактор] Финальный текст готов ({len(response.content)} символов)")
    return {"final_text": response.content}


writing_graph = StateGraph(WritingState)
writing_graph.add_node("drafter", drafter)
writing_graph.add_node("editor", editor)
writing_graph.add_edge(START, "drafter")
writing_graph.add_edge("drafter", "editor")
writing_graph.add_edge("editor", END)
writing_compiled = writing_graph.compile()

## Верхний супервайзер

Верхний супервайзер не знает внутреннего устройства команд — он оперирует только их именами: `research_team` и `writing_team`. На каждом шаге он оценивает текущий статус (есть ли исследование, есть ли текст) и решает, какую команду вызвать следующей или пора ли завершить работу. Обратите внимание на fallback-логику в `except`: если LLM вернёт невалидный JSON, супервайзер всё равно извлечёт решение из текста.

In [5]:
class TopState(TypedDict):
    task: str
    research: str
    draft: str
    final_text: str
    next_team: str


def top_supervisor(state: TopState) -> dict:
    """Верхний супервайзер: координирует команды."""
    status_parts = []
    if state.get("research"):
        status_parts.append(f"Исследование: готово ({len(state['research'])} симв.)")
    else:
        status_parts.append("Исследование: нет")
    if state.get("final_text"):
        status_parts.append(f"Финальный текст: готов ({len(state['final_text'])} симв.)")
    elif state.get("draft"):
        status_parts.append(f"Черновик: есть ({len(state['draft'])} симв.)")
    else:
        status_parts.append("Текст: нет")

    status = "\n".join(status_parts)

    response = llm.invoke([
        SystemMessage(content="""Ты — главный супервайзер. Управляешь двумя командами:
- research_team: собирает факты по теме (вызывай ПЕРВОЙ)
- writing_team: пишет статью на основе исследования (вызывай ВТОРОЙ)
- FINISH: работа завершена (когда есть финальный текст)

Верни JSON (и ТОЛЬКО JSON): {"next": "research_team"} или {"next": "writing_team"} или {"next": "FINISH"}"""),
        HumanMessage(content=f"Задача: {state['task']}\n\nСтатус:\n{status}")
    ])

    try:
        parsed = json.loads(response.content.strip())
        next_team = parsed.get("next", "FINISH")
    except json.JSONDecodeError:
        content = response.content.lower()
        if "research" in content:
            next_team = "research_team"
        elif "writ" in content:
            next_team = "writing_team"
        else:
            next_team = "FINISH"

    print(f"  [Гл. супервайзер] -> {next_team}")
    return {"next_team": next_team}


def route_teams(state: TopState) -> str:
    next_team = state.get("next_team", "FINISH")
    if next_team == "FINISH":
        return END
    return next_team

## Сборка верхнего графа

Скомпилированные подграфы добавляются как обычные узлы через `add_node`. LangGraph автоматически сопоставляет поля между `TopState` и состояниями команд по совпадению имён: например, поле `task` из `TopState` попадёт в `ResearchState.task`, а `research` из результата подграфа вернётся обратно в `TopState.research`. Установим `recursion_limit=40`, потому что в системе два уровня рекурсии.

In [6]:
top_graph = StateGraph(TopState)
top_graph.add_node("supervisor", top_supervisor)
top_graph.add_node("research_team", research_compiled)
top_graph.add_node("writing_team", writing_compiled)

top_graph.add_edge(START, "supervisor")
top_graph.add_conditional_edges("supervisor", route_teams, {
    "research_team": "research_team",
    "writing_team": "writing_team",
    END: END,
})
top_graph.add_edge("research_team", "supervisor")
top_graph.add_edge("writing_team", "supervisor")

app = top_graph.compile()

## Запуск

In [7]:
result = app.invoke({
    "task": "Обзор архитектурных паттернов мультиагентных систем",
    "research": "", "draft": "", "final_text": "", "next_team": "",
}, {"recursion_limit": 40})

print(f"\n  Исследование: {'есть' if result['research'] else 'нет'} ({len(result.get('research', ''))} симв.)")
print(f"  Финальный текст: {result.get('final_text', 'Нет')[:200]}...")

  [Гл. супервайзер] -> research_team


  [Веб-исследователь] Факты найдены (1331 символов)


  [Документалист] Дополнено (1806 символов)


  [Гл. супервайзер] -> writing_team


  [Автор] Черновик готов (1125 символов)


  [Редактор] Финальный текст готов (1205 символов)


  [Гл. супервайзер] -> FINISH

  Исследование: есть (3156 симв.)
  Финальный текст: Мультиагентные системы можно строить по разным архитектурным паттернам, и выбор зависит от того, что важнее: управляемость, масштабируемость или гибкость. В централизованной архитектуре один управляющ...
